# 01 - EDA: Concrete Compressive Strength
Goal: understand the data before modeling. Step 1 = load, clean column names, sanity checks.

In [ ]:
import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/concrete.csv"
df_raw = pd.read_csv(RAW_PATH)
df_raw.shape

In [ ]:
# Original names are long and messy (double spaces, units, trailing space).
# Position-based rename is safest: the column ORDER is fixed by the dataset.
COLS = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_agg", "fine_agg", "age_days", "strength_mpa",
]
assert len(COLS) == df_raw.shape[1], "column count mismatch"
df = df_raw.copy()
df.columns = COLS
df.sample(5)

In [ ]:
# Sanity checks: types, missing values, duplicates
print(df.dtypes, "\n")
print("missing per column:\n", df.isnull().sum(), "\n")
print("duplicate rows:", df.duplicated().sum())

**Finding:** 25 exact duplicate rows. **Decision:** remove them (keep first) before any train/test split, so identical rows cannot leak into the test set. The dedupe below is in-memory only, so the rest of EDA is not skewed by repeats. The permanent cleaning will live in `src/features.py`.

In [ ]:
# Look before deleting: keep=False shows EVERY copy (original + twins)
dups = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print(len(dups), "rows involved in duplication")
dups.head(10)

In [ ]:
n_before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(n_before, "->", len(df), "| removed:", n_before - len(df))
assert df.duplicated().sum() == 0